In [ ]:
# LA Parking Citations — Exploration

Loading `Parking_Citations_20260426.csv` (~6.2 GB) with [Polars](https://pola.rs/).

The dataset has mixed types:
- **Strings**: plate state, make, color, violation description, location, etc.
- **Integers**: fine amount, agency code
- **Floats**: latitude, longitude
- **Datetimes**: issue date
- **Coordinates / geospatial**: `loc_lat` + `loc_long`, plus a WKT `POINT` in `geocodelocation`

We use `pl.scan_csv` (lazy) so we only materialize what we ask for.

In [1]:
from pathlib import Path

import polars as pl

CSV_PATH = Path("Parking_Citations_20260426.csv")
print(f"polars {pl.__version__} — file size: {CSV_PATH.stat().st_size / 1e9:.2f} GB")

polars 1.40.1 — file size: 6.22 GB


In [2]:
SCHEMA: dict[str, pl.DataType] = {
    "ticket_number":         pl.String,
    "issue_date":            pl.String,
    "issue_time":            pl.String,
    "meter_id":              pl.String,
    "marked_time":           pl.String,
    "rp_state_plate":        pl.String,
    "plate_expiry_date":     pl.String,
    "vin":                   pl.String,
    "make":                  pl.String,
    "body_style":            pl.String,
    "color":                 pl.String,
    "location":              pl.String,
    "route":                 pl.String,
    "agency":                pl.Int32,
    "violation_code":        pl.String,
    "violation_description": pl.String,
    "fine_amount":           pl.Float64,
    "agency_desc":           pl.String,
    "color_desc":            pl.String,
    "body_style_desc":       pl.String,
    "loc_lat":               pl.Float64,
    "loc_long":              pl.Float64,
    "geocodelocation":       pl.String,
}

lf = pl.scan_csv(
    CSV_PATH,
    schema_overrides=SCHEMA,
    null_values=["", "NA", "N/A"],
    ignore_errors=True,
).with_columns(
    pl.col("issue_date").str.strptime(
        pl.Datetime, format="%Y %b %d %I:%M:%S %p", strict=False
    ),
)

lf.collect_schema()

Schema([('ticket_number', String),
        ('issue_date', Datetime(time_unit='us', time_zone=None)),
        ('issue_time', String),
        ('meter_id', String),
        ('marked_time', String),
        ('rp_state_plate', String),
        ('plate_expiry_date', String),
        ('vin', String),
        ('make', String),
        ('body_style', String),
        ('color', String),
        ('location', String),
        ('route', String),
        ('agency', Int32),
        ('violation_code', String),
        ('violation_description', String),
        ('fine_amount', Float64),
        ('agency_desc', String),
        ('color_desc', String),
        ('body_style_desc', String),
        ('loc_lat', Float64),
        ('loc_long', Float64),
        ('geocodelocation', String)])

## Preview

`scan_csv` returns a `LazyFrame` — nothing has been loaded yet. We call `.head().collect()` to materialize just the first few rows.

In [3]:
df_head = lf.head(10).collect()
df_head

ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation
str,datetime[μs],str,str,str,str,str,str,str,str,str,str,str,i32,str,str,f64,str,str,str,f64,f64,str
"""4602073232""",2025-04-26 00:00:00,"""904""",null,"""0000""","""CA""","""202512""",null,"""FORD""","""PA""","""WT""","""1875 20TH ST W""",null,55,"""22500H""","""DOUBLE PARKING""",68.0,"""55 - DOT - SOUTHERN""","""WHITE""","""PASSENGER CAR""",34.038267,-118.299593,"""POINT (-118.29959251 34.038266…"
"""4601302834""",2025-04-26 00:00:00,"""830""",null,"""0000""","""CA""","""202510""",null,"""CHEV""","""PA""","""SL""","""5100 WOODMAN AVE""",null,53,"""22514""","""FIRE HYDRANT""",68.0,"""53 - DOT - VALLEY""","""SILVER""","""PASSENGER CAR""",34.16337,-118.431059,"""POINT (-118.4310588 34.16337)"""
"""4601978496""",2025-04-26 00:00:00,"""825""","""WU988""","""0000""","""CA""","""202508""",null,"""TOYT""","""PA""","""SL""","""601 SAINT PAUL AV""",null,56,"""88.13B+""","""METER EXP.""",63.0,"""56 - DOT - CENTRAL""","""SILVER""","""PASSENGER CAR""",34.052868,-118.260923,"""POINT (-118.260923 34.05286752…"
"""4602159520""",2025-04-26 00:00:00,"""935""",null,"""0000""","""OR""","""202606""",null,"""CHEV""","""PU""","""BK""","""25828 PRESIDENT AVE""",null,55,"""80.61""","""STANDNG IN ALLEY""",68.0,"""55 - DOT - SOUTHERN""","""BLACK""","""PICK-UP TRUCK""",33.788801,-118.304095,"""POINT (-118.30409538 33.788800…"
"""4602061811""",2025-04-26 00:00:00,"""1,255""",null,"""0000""","""CA""","""202406""",null,"""NISS""","""PA""",null,"""7620 VARIEL AVE""",null,53,"""80.73.2""","""EXCEED 72HRS-ST""",68.0,"""53 - DOT - VALLEY""",null,"""PASSENGER CAR""",34.208873,-118.592801,"""POINT (-118.59280069 34.208873…"
"""4601664760""",2025-04-26 00:00:00,"""942""","""WA126""","""0000""","""CA""","""202504""",null,"""DODG""","""PU""","""GY""","""608 WESTLAKE AV S""",null,56,"""88.13B+""","""METER EXP.""",63.0,"""56 - DOT - CENTRAL""","""GREY""","""PICK-UP TRUCK""",34.058402,-118.274051,"""POINT (-118.27405053 34.058401…"
"""4602043423""",2025-04-26 00:00:00,"""1,118""",null,"""0000""","""CA""","""202605""",null,"""FORD""","""PA""","""RD""","""2303 CHARLOTTE ST""",null,56,"""80.69AP+""","""NO STOP/STANDING""",93.0,"""56 - DOT - CENTRAL""","""RED""","""PASSENGER CAR""",34.056803,-118.202765,"""POINT (-118.20276478 34.056803…"
"""4602639440""",2025-04-26 00:00:00,"""1,244""",null,"""0000""","""CA""","""202504""",null,"""TOYT""","""PA""","""SL""","""1700 BARNETT ROAD""",null,56,"""80.61""","""STANDNG IN ALLEY""",68.0,"""56 - DOT - CENTRAL""","""SILVER""","""PASSENGER CAR""",34.061847,-118.175331,"""POINT (-118.17533122 34.061846…"
"""4602094615""",2025-04-26 00:00:00,"""920""",null,"""0000""","""CA""","""0""",null,"""BUIC""","""PA""","""BK""","""325 6TH ST W""",null,56,"""80.56E2""","""YELLOW ZONE""",58.0,"""56 - DOT - CENTRAL""","""BLACK""","""PASSENGER CAR""",34.04697,-118.252961,"""POINT (-118.25296108 34.046970…"


In [4]:
df_head.schema

Schema([('ticket_number', String),
        ('issue_date', Datetime(time_unit='us', time_zone=None)),
        ('issue_time', String),
        ('meter_id', String),
        ('marked_time', String),
        ('rp_state_plate', String),
        ('plate_expiry_date', String),
        ('vin', String),
        ('make', String),
        ('body_style', String),
        ('color', String),
        ('location', String),
        ('route', String),
        ('agency', Int32),
        ('violation_code', String),
        ('violation_description', String),
        ('fine_amount', Float64),
        ('agency_desc', String),
        ('color_desc', String),
        ('body_style_desc', String),
        ('loc_lat', Float64),
        ('loc_long', Float64),
        ('geocodelocation', String)])

## Geospatial columns

The CSV has two flavors of location data:

1. `loc_lat` / `loc_long` — already typed as `Float64`. Use these for any numeric/spatial filtering inside Polars.
2. `geocodelocation` — a [WKT](https://en.wikipedia.org/wiki/Well-known_text_representation_of_geometry) string like `POINT (-118.29959251 34.0382668)`. Polars keeps it as text; if you need real geometry ops (intersections, buffers, projections), pair it with `shapely` / `geopandas` (`pip install shapely geopandas`).

Quick demo: count citations issued inside a rough bounding box around downtown LA.

In [5]:
downtown_la = (
    lf.filter(
        pl.col("loc_lat").is_between(34.03, 34.07)
        & pl.col("loc_long").is_between(-118.27, -118.23)
    )
    .select(pl.len().alias("citations_in_dtla"))
    .collect()
)
downtown_la

citations_in_dtla
u32
3174199


## Building a queryable SQLite DB

The full lazy scan above is great for one-off analyses but it has to re-read the 6.2 GB CSV every time. For day-to-day querying — and to keep the dataset fresh from the [Socrata API](https://dev.socrata.com/foundry/data.lacity.org/4f5p-udkv) — we ingest the CSV once into a local SQLite file, then top it up with newest-first API calls.

The logic lives in [`parking_db.py`](parking_db.py); the cells below show end-to-end usage.

**Workflow**

1. `parking_db.init_db()` — create the schema (idempotent).
2. `parking_db.bulk_load_csv(...)` — stream the CSV in 100k-row batches; uses `INSERT OR IGNORE` so it's safe to re-run.
3. `parking_db.update_from_api(...)` — paginate Socrata sorted by `:updated_at DESC` and stop as soon as a page contains a `ticket_number` we already have.
4. `parking_db.db_stats(...)` — quick health check.

> The bulk load is the slow step (≈25 M rows, expect a few minutes). After that, `update_from_api` is fast — usually only a page or two.

In [ ]:
import parking_db

DB_PATH = "parking_citations.db"
parking_db.init_db(DB_PATH)
parking_db.db_stats(DB_PATH)

### One-time bulk load

⚠️ This will take a few minutes (≈25 M rows). Re-running is safe — the primary key on `ticket_number` plus `INSERT OR IGNORE` makes it idempotent.

In [ ]:
parking_db.bulk_load_csv(CSV_PATH, DB_PATH, batch_size=100_000)
parking_db.db_stats(DB_PATH)

### Incremental sync from the API

Walks the dataset newest-first (`:updated_at DESC`). On each page it checks whether any `ticket_number` is already in the DB; the first such hit means we're caught up and we stop.

The Socrata app token is read automatically from `.env` (env var `SOCRATA_APP_TOKEN`) — drop your token into the `.env` file in this folder and you're done. You can still pass `app_token="..."` explicitly to override.

In [ ]:
import os

print("token loaded:", bool(os.getenv("SOCRATA_APP_TOKEN")))

parking_db.update_from_api(DB_PATH, page_size=1000)

### Querying the DB with Polars

Once the data is in SQLite you can pull arbitrary slices straight into a Polars DataFrame via `pl.read_database_uri`. SQLite returns results in milliseconds for indexed lookups.

In [ ]:
import sqlite3

with sqlite3.connect(DB_PATH) as conn:
    top_violations = pl.read_database(
        """
        SELECT violation_description, COUNT(*) AS n,
               ROUND(AVG(fine_amount), 2) AS avg_fine
        FROM citations
        WHERE violation_description IS NOT NULL
        GROUP BY violation_description
        ORDER BY n DESC
        LIMIT 15
        """,
        connection=conn,
    )
top_violations